# Model Evaluation for UNet EO Segmentation

This notebook evaluates the single-timestep CNN UNet segmentation model
using IoU, precision, recall, and F1 score on TFRecord data.

It replaces the Prithvi temporal transformer from the original evaluation
notebook with the UNet architecture used in
`Training_and_Segmentation_UNet_Clean.ipynb`. The dataset selects only one
timestep per sample: the latest valid frame, falling back to the final
recorded timestep if no valid frame exists.


## Environment

The evaluation workflow uses TensorFlow only to parse TFRecords and PyTorch
for model inference. TensorFlow GPU access is disabled in the imports cell
so it does not reserve CUDA memory needed by PyTorch.


In [ ]:
%pip install "numpy<2" "opencv-python==4.11.0.86" "opencv-python-headless==4.11.0.86" "albumentations==1.4.6" "rasterio==1.4.4"


In [1]:
import torch

In [2]:
import importlib.util
import os
import random
import struct
from pathlib import Path

# Rasterio's wheel contains the PROJ database that matches its PROJ runtime.
_rasterio_spec = importlib.util.find_spec("rasterio")
if _rasterio_spec is not None:
    _proj_data = Path(_rasterio_spec.origin).parent / "proj_data"
    if _proj_data.exists():
        os.environ["PROJ_DATA"] = str(_proj_data)
        os.environ["PROJ_LIB"] = str(_proj_data)

import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

# Use TensorFlow only for TFRecord parsing; keep GPU memory for PyTorch.
try:
    tf.config.set_visible_devices([], "GPU")
    print("TensorFlow GPU disabled.")
except RuntimeError as error:
    print(
        "TensorFlow GPU was already initialized. "
        "Restart the kernel and run this cell before any TensorFlow work."
    )
    print(error)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True


TRAIN_TFRECORD = Path(
    r"D:\Data\ssm_temporal\ssm_footprint_train_new.tfrecord"
)
VAL_TFRECORD = Path(
    r"D:\Data\ssm_temporal\ssm_footprint_no_cloud_val.tfrecord"
)

PREFERRED_MODEL_PATH = Path("models/unet_state_dict_clean.pt")
FALLBACK_MODEL_PATH = Path("models/unet_state_dict.pt")
MODEL_PATH = (
    PREFERRED_MODEL_PATH
    if PREFERRED_MODEL_PATH.exists()
    else FALLBACK_MODEL_PATH
)

IMAGE_SIZE = 224
IN_CHANNELS = 6
NUM_CLASSES = 3
BASE_CHANNELS = 64
BATCH_SIZE = 8
NUM_WORKERS = 0
EVALUATE_TRAIN_SET = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("PROJ data:", os.environ.get("PROJ_DATA"))
for label, path in {
    "train data": TRAIN_TFRECORD,
    "validation data": VAL_TFRECORD,
    "model checkpoint": MODEL_PATH,
}.items():
    print(f"{label:20s}: {path} ({'found' if path.exists() else 'missing'})")


TensorFlow GPU disabled.
Device: cuda
PROJ data: c:\Users\emmanuelasare\AppData\Local\anaconda3\envs\ml-env2\lib\site-packages\rasterio\proj_data
train data          : D:\Data\ssm_temporal\ssm_footprint_train_new.tfrecord (found)
validation data     : D:\Data\ssm_temporal\ssm_footprint_no_cloud_val.tfrecord (found)
model checkpoint    : unet_state_dict_clean.pt (found)


c:\Users\emmanuelasare\AppData\Local\anaconda3\envs\ml-env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset

The dataset mirrors the UNet training notebook: it reads the temporal
TFRecord, selects one latest valid timestep, normalizes that frame, and
returns `image` as `(C, H, W)` for CNN inference.


In [3]:
def _center_pad_hw(array, target_hw):
    target_h, target_w = target_hw
    h, w = array.shape[:2]
    pad_h = max(target_h - h, 0)
    pad_w = max(target_w - w, 0)
    pad_top = pad_h // 2
    pad_left = pad_w // 2
    padding = [
        (pad_top, pad_h - pad_top),
        (pad_left, pad_w - pad_left),
    ]
    padding.extend([(0, 0)] * (array.ndim - 2))
    return np.pad(array, padding, mode="constant")


def _center_pad_chw(image, target_hw):
    channels, h, w = image.shape
    image_hwc = image.transpose(1, 2, 0)
    image_hwc = _center_pad_hw(image_hwc, target_hw)
    padded_h, padded_w, _ = image_hwc.shape
    return image_hwc.transpose(2, 0, 1).reshape(
        channels, padded_h, padded_w
    )


class MineFootprintTFRecordDataset(Dataset):
    MEAN = np.array(
        [1087.0, 1342.0, 1433.0, 2734.0, 1958.0, 1363.0],
        dtype=np.float32,
    )
    STD = np.array(
        [2248.0, 2179.0, 2178.0, 1850.0, 1242.0, 1049.0],
        dtype=np.float32,
    )

    FEATURE_DESCRIPTION = {
        "image_raw": tf.io.FixedLenFeature([], tf.string),
        "mask_raw": tf.io.FixedLenFeature([], tf.string),
        "height": tf.io.FixedLenFeature([], tf.int64),
        "width": tf.io.FixedLenFeature([], tf.int64),
        "channels": tf.io.FixedLenFeature([], tf.int64),
        "timesteps": tf.io.FixedLenFeature([], tf.int64),
        "temporal_coords": tf.io.VarLenFeature(tf.float32),
        "location_coords": tf.io.FixedLenFeature([2], tf.float32),
    }

    def __init__(self, tfrecord_path, transform=None, pad_to=(224, 224)):
        self.tfrecord_path = Path(tfrecord_path)
        if not self.tfrecord_path.exists():
            raise FileNotFoundError(
                f"TFRecord not found: {self.tfrecord_path}. "
                "Update the path in the configuration cell."
            )
        self.transform = transform
        self.pad_to = pad_to
        self._offsets = self._scan_index()
        self._file = self.tfrecord_path.open("rb")

    def _scan_index(self):
        offsets = []
        with self.tfrecord_path.open("rb") as file:
            position = 0
            while True:
                header = file.read(12)
                if not header:
                    break
                record_length = struct.unpack("<Q", header[:8])[0]
                offsets.append(position)
                position += 12 + record_length + 4
                file.seek(position)
        return offsets

    def _read_record(self, offset):
        self._file.seek(offset)
        header = self._file.read(12)
        record_length = struct.unpack("<Q", header[:8])[0]
        data = self._file.read(record_length)
        self._file.read(4)
        return data

    def __len__(self):
        return len(self._offsets)

    def _target_timestep_index(self, image):
        temporal_mask = (
            np.abs(image).sum(axis=(0, 2, 3)) > 0
        ).astype(np.float32)
        valid = np.flatnonzero(temporal_mask > 0)
        if len(valid):
            return int(valid[-1]), temporal_mask
        return image.shape[1] - 1, temporal_mask

    def _normalize_image(self, image, is_valid=True):
        channels = image.shape[0]
        normalized = np.zeros_like(image, dtype=np.float32)
        if not is_valid:
            return normalized

        if channels == len(self.MEAN):
            mean = self.MEAN.reshape(channels, 1, 1)
            std = (self.STD + 1e-6).reshape(channels, 1, 1)
        else:
            mean = image.mean(axis=(1, 2), keepdims=True)
            std = image.std(axis=(1, 2), keepdims=True) + 1e-6

        normalized = (image - mean) / std
        return normalized.astype(np.float32)

    def _apply_transform(self, image, mask):
        image_hwc = image.transpose(1, 2, 0)
        augmented = self.transform(image=image_hwc, mask=mask)
        image_hwc = np.asarray(augmented["image"])
        mask = np.asarray(augmented["mask"])
        image = image_hwc.transpose(2, 0, 1)
        return image, mask

    def __getitem__(self, index):
        serialized = self._read_record(self._offsets[index])
        example = tf.io.parse_single_example(
            serialized, self.FEATURE_DESCRIPTION
        )

        h = int(example["height"])
        w = int(example["width"])
        channels = int(example["channels"])
        timesteps = int(example["timesteps"])

        temporal_image = np.frombuffer(
            example["image_raw"].numpy(), dtype=np.float32
        ).reshape(channels, timesteps, h, w)
        mask = np.frombuffer(
            example["mask_raw"].numpy(), dtype=np.uint8
        ).reshape(h, w)

        temporal_image = np.nan_to_num(temporal_image, nan=0.0)
        mask = np.nan_to_num(mask, nan=0).astype(np.uint8)

        target_timestep, temporal_mask = self._target_timestep_index(
            temporal_image
        )
        image = temporal_image[:, target_timestep]
        image = self._normalize_image(
            image, is_valid=bool(temporal_mask[target_timestep] > 0)
        )
        image = _center_pad_chw(image, self.pad_to)
        mask = _center_pad_hw(mask, self.pad_to)

        temporal_coords = tf.sparse.to_dense(
            example["temporal_coords"]
        ).numpy().astype(np.float32).reshape(timesteps, 2)
        target_temporal_coords = temporal_coords[target_timestep]
        location_coords = (
            example["location_coords"].numpy().astype(np.float32)
        )

        if self.transform is not None:
            image, mask = self._apply_transform(image, mask)

        return {
            "image": torch.from_numpy(
                np.ascontiguousarray(image)
            ).float(),
            "temporal_coords": torch.from_numpy(
                np.ascontiguousarray(target_temporal_coords)
            ).float(),
            "location_coords": torch.from_numpy(
                np.ascontiguousarray(location_coords)
            ).float(),
            "target_timestep": torch.tensor(
                target_timestep, dtype=torch.long
            ),
            "mask": torch.from_numpy(
                np.ascontiguousarray(mask)
            ).long(),
        }

    def close(self):
        if hasattr(self, "_file") and not self._file.closed:
            self._file.close()

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass


In [10]:
EVALUATE_TRAIN_SET = True

In [11]:
val_dataset = MineFootprintTFRecordDataset(
    VAL_TFRECORD,
    transform=None,
    pad_to=(IMAGE_SIZE, IMAGE_SIZE),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
)

train_dataset = None
train_loader = None
if EVALUATE_TRAIN_SET:
    train_dataset = MineFootprintTFRecordDataset(
        TRAIN_TFRECORD,
        transform=None,
        pad_to=(IMAGE_SIZE, IMAGE_SIZE),
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
    )

first_batch = next(iter(val_loader))
for key, value in first_batch.items():
    print(f"{key:20s}: {tuple(value.shape)}")


image               : (8, 6, 224, 224)
temporal_coords     : (8, 2)
location_coords     : (8, 2)
target_timestep     : (8,)
mask                : (8, 224, 224)


## UNet Model

This is the same CNN UNet definition used by
`Training_and_Segmentation_UNet_Clean.ipynb`.


In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, image):
        return self.block(image)


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            ConvBlock(in_channels, out_channels, dropout=dropout),
        )

    def forward(self, image):
        return self.block(image)


class UpBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        bilinear=True,
        dropout=0.0,
    ):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(
                scale_factor=2,
                mode="bilinear",
                align_corners=False,
            )
            conv_in_channels = in_channels + skip_channels
        else:
            self.up = nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size=2,
                stride=2,
            )
            conv_in_channels = out_channels + skip_channels

        self.conv = ConvBlock(
            conv_in_channels, out_channels, dropout=dropout
        )

    def forward(self, image, skip):
        image = self.up(image)
        diff_y = skip.size(2) - image.size(2)
        diff_x = skip.size(3) - image.size(3)
        image = F.pad(
            image,
            [
                diff_x // 2,
                diff_x - diff_x // 2,
                diff_y // 2,
                diff_y - diff_y // 2,
            ],
        )
        image = torch.cat([skip, image], dim=1)
        return self.conv(image)


class UNetSegmentation(nn.Module):
    def __init__(
        self,
        in_channels=6,
        num_classes=3,
        base_channels=64,
        bilinear=True,
        dropout=0.0,
    ):
        super().__init__()
        self.input = ConvBlock(
            in_channels, base_channels, dropout=dropout
        )
        self.down1 = DownBlock(
            base_channels, base_channels * 2, dropout=dropout
        )
        self.down2 = DownBlock(
            base_channels * 2, base_channels * 4, dropout=dropout
        )
        self.down3 = DownBlock(
            base_channels * 4, base_channels * 8, dropout=dropout
        )
        self.down4 = DownBlock(
            base_channels * 8, base_channels * 16, dropout=dropout
        )
        self.up1 = UpBlock(
            base_channels * 16,
            base_channels * 8,
            base_channels * 8,
            bilinear=bilinear,
            dropout=dropout,
        )
        self.up2 = UpBlock(
            base_channels * 8,
            base_channels * 4,
            base_channels * 4,
            bilinear=bilinear,
            dropout=dropout,
        )
        self.up3 = UpBlock(
            base_channels * 4,
            base_channels * 2,
            base_channels * 2,
            bilinear=bilinear,
            dropout=dropout,
        )
        self.up4 = UpBlock(
            base_channels * 2,
            base_channels,
            base_channels,
            bilinear=bilinear,
            dropout=dropout,
        )
        self.head = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, image):
        x1 = self.input(image)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        output = self.up1(x5, x4)
        output = self.up2(output, x3)
        output = self.up3(output, x2)
        output = self.up4(output, x1)
        return self.head(output)


def build_unet_segmentation(
    num_classes=3,
    in_channels=6,
    base_channels=64,
    bilinear=True,
    dropout=0.0,
):
    return UNetSegmentation(
        in_channels=in_channels,
        num_classes=num_classes,
        base_channels=base_channels,
        bilinear=bilinear,
        dropout=dropout,
    )


In [6]:
def read_state_dict(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        checkpoint = checkpoint["state_dict"]
    return checkpoint


def clean_state_dict_keys(state):
    prefixes = (
        "module.",
        "model.",
        "net.",
        "unet.",
    )
    cleaned = {}
    for raw_key, value in state.items():
        key = raw_key
        changed = True
        while changed:
            changed = False
            for prefix in prefixes:
                if key.startswith(prefix):
                    key = key[len(prefix):]
                    changed = True
        cleaned[key] = value
    return cleaned


def load_model_state(model, checkpoint_path, device):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Model checkpoint not found: {checkpoint_path}"
        )

    state = clean_state_dict_keys(read_state_dict(checkpoint_path))
    try:
        model.load_state_dict(state, strict=True)
    except RuntimeError as error:
        print("Strict checkpoint load failed.")
        print("First checkpoint keys:", list(state.keys())[:10])
        raise error

    model.to(device)
    model.eval()
    print(f"Loaded model from {checkpoint_path}")


model = build_unet_segmentation(
    num_classes=NUM_CLASSES,
    in_channels=IN_CHANNELS,
    base_channels=BASE_CHANNELS,
    bilinear=True,
    dropout=0.0,
).to(device)

load_model_state(model, MODEL_PATH, device)

trainable = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
total = sum(parameter.numel() for parameter in model.parameters())
print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters:     {total:,}")


Loaded model from unet_state_dict_clean.pt
Trainable parameters: 31,386,691
Total parameters:     31,386,691


## Metrics

Metrics are accumulated over every pixel in the loader. `NaN` means the
class had no denominator for that metric, such as no predictions and no
ground-truth pixels for that class.


In [7]:
def calculate_raw_scores(ground_truth_mask, predicted_mask, num_classes):
    true_positives = {}
    false_positives = {}
    false_negatives = {}
    for class_id in range(num_classes):
        true_positives[class_id] = np.sum(
            (ground_truth_mask == class_id)
            & (predicted_mask == class_id)
        )
        false_positives[class_id] = np.sum(
            (ground_truth_mask != class_id)
            & (predicted_mask == class_id)
        )
        false_negatives[class_id] = np.sum(
            (ground_truth_mask == class_id)
            & (predicted_mask != class_id)
        )
    return true_positives, false_positives, false_negatives


def calc_f1_score(precision, recall):
    if np.isnan(precision) or np.isnan(recall):
        return np.nan
    if precision + recall == 0:
        return np.nan
    return (2 * precision * recall) / (precision + recall)


@torch.inference_mode()
def predict_masks(model, images, device):
    model.eval()
    logits = model(images.to(device))
    return logits.argmax(dim=1).cpu().numpy()


@torch.inference_mode()
def calculate_mean_precision_recall(
    model,
    loader,
    num_classes,
    device,
    description="Evaluating",
):
    total_true_positives = {c: 0 for c in range(num_classes)}
    total_false_positives = {c: 0 for c in range(num_classes)}
    total_false_negatives = {c: 0 for c in range(num_classes)}
    correct = 0
    pixels = 0

    for batch in tqdm(loader, desc=description, unit="batch"):
        images = batch["image"]
        masks = batch["mask"].cpu().numpy()
        predictions = predict_masks(model, images, device)

        correct += (predictions == masks).sum()
        pixels += masks.size

        for prediction, label in zip(predictions, masks):
            tp, fp, fn = calculate_raw_scores(
                label, prediction, num_classes
            )
            for class_id in range(num_classes):
                total_true_positives[class_id] += tp[class_id]
                total_false_positives[class_id] += fp[class_id]
                total_false_negatives[class_id] += fn[class_id]

    rows = []
    for class_id in range(num_classes):
        tp = total_true_positives[class_id]
        fp = total_false_positives[class_id]
        fn = total_false_negatives[class_id]

        precision = tp / (tp + fp) if (tp + fp) else np.nan
        recall = tp / (tp + fn) if (tp + fn) else np.nan
        iou = tp / (tp + fp + fn) if (tp + fp + fn) else np.nan
        f1 = calc_f1_score(precision, recall)

        rows.append(
            {
                "class_val": class_id,
                "iou": iou,
                "precision": precision,
                "recall": recall,
                "f1 score": f1,
                "true_positive_pixels": tp,
                "false_positive_pixels": fp,
                "false_negative_pixels": fn,
            }
        )

    summary = pd.DataFrame(rows)
    summary.attrs["pixel_accuracy"] = correct / max(pixels, 1)
    return summary


## Run Evaluation

The main output is the validation metrics table. Enable
`EVALUATE_TRAIN_SET` in the configuration cell if you also want train-set
metrics.


In [8]:
val_results_df = calculate_mean_precision_recall(
    model,
    val_loader,
    NUM_CLASSES,
    device,
    description="Validation",
)
print(f"Validation pixel accuracy: {val_results_df.attrs['pixel_accuracy']:.2%}")
val_results_df


Validation: 100%|██████████| 318/318 [01:46<00:00,  2.97batch/s]

Validation pixel accuracy: 99.02%


,class_val,iou,precision,recall,f1 score,true_positive_pixels,false_positive_pixels,false_negative_pixels
0,0,0.991386,0.993448,0.997911,0.995675,124600957,821825,260776
1,1,0.570179,0.838788,0.640353,0.726260,1567083,301187,880134
2,2,0.130196,0.217177,0.245326,0.230395,33877,122111,104213


In [12]:
if train_loader is not None:
    train_results_df = calculate_mean_precision_recall(
        model,
        train_loader,
        NUM_CLASSES,
        device,
        description="Train",
    )
    print(f"Train pixel accuracy: {train_results_df.attrs['pixel_accuracy']:.2%}")
    display(train_results_df)


Train: 100%|██████████| 2669/2669 [14:59<00:00,  2.97batch/s]

Train pixel accuracy: 98.47%


,class_val,iou,precision,recall,f1 score,true_positive_pixels,false_positive_pixels,false_negative_pixels
0,0,0.987347,0.990077,0.997215,0.993633,1041750232,10440612,2909889
1,1,0.465267,0.712883,0.572558,0.635061,11870832,4781033,8862148
2,2,0.170031,0.507233,0.203675,0.290644,1174011,1140528,4590136


## Visual Check

A quick qualitative check helps catch class-index or normalization issues
that aggregate metrics can hide.


In [ ]:
CLASS_COLORS = {
    0: [0, 0, 0],
    1: [251, 72, 196],
    2: [180, 96, 0],
    3: [255, 255, 0],
}


def mask_to_rgb(mask, colors=CLASS_COLORS):
    mask = np.asarray(mask)
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_id, color in colors.items():
        rgb[mask == class_id] = color
    return rgb


def single_rgb(image, dataset, rgb_indices=(2, 1, 0), divisor=3000.0):
    image = torch.as_tensor(image).detach().cpu().numpy()
    channels = image.shape[0]

    if channels == len(dataset.MEAN):
        mean = np.asarray(dataset.MEAN, dtype=np.float32).reshape(
            channels, 1, 1
        )
        std = np.asarray(dataset.STD, dtype=np.float32).reshape(
            channels, 1, 1
        )
        image = image * std + mean
        rgb = image[list(rgb_indices)].transpose(1, 2, 0)
        return np.clip(rgb / divisor, 0.0, 1.0)

    rgb = image[:3].transpose(1, 2, 0)
    rgb_min = rgb.min()
    rgb_max = rgb.max()
    return (rgb - rgb_min) / (rgb_max - rgb_min + 1e-6)


@torch.inference_mode()
def visualize_prediction(model, dataset, device, index=None):
    if index is None:
        index = random.randrange(len(dataset))

    sample = dataset[index]
    image = sample["image"].unsqueeze(0)
    logits = model(image.to(device))
    prediction = logits.argmax(dim=1)[0].cpu().numpy()
    ground_truth = sample["mask"].cpu().numpy()
    input_rgb = single_rgb(sample["image"], dataset)
    coords = sample["temporal_coords"].cpu().numpy()

    figure, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    axes[0].imshow(input_rgb)
    if np.any(coords):
        year, day = coords
        axes[0].set_title(
            f"Input RGB ({int(year)} | DOY {int(day)})"
        )
    else:
        axes[0].set_title("Input RGB")
    axes[1].imshow(mask_to_rgb(ground_truth))
    axes[1].set_title("Ground truth")
    axes[2].imshow(mask_to_rgb(prediction))
    axes[2].set_title("Prediction")

    for axis in axes:
        axis.axis("off")
    figure.suptitle(f"Validation sample {index}")
    plt.tight_layout()
    plt.show()


In [ ]:
visualize_prediction(
    model,
    val_dataset,
    device=device,
    index=min(1000, len(val_dataset) - 1),
)


## CUDA Memory Check

This optional cell confirms PyTorch memory use. If `nvidia-smi` is much
higher than this summary, another library or process is reserving GPU memory.


In [ ]:
if torch.cuda.is_available():
    print(torch.cuda.memory_summary(device=device))
